# 05 SurgeDesk evaluation and benchmark
Read-only views of outputs produced by `evaluate.py` and `benchmark.py`. Missing results stay explicitly unavailable; this notebook performs no training, inference, or benchmarking.

In [1]:
# Bootstrap: make the repo importable whether the notebook runs in IBM's container or ours
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from surgedesk.envcheck import run
ENV = run()

SurgeDesk environment check
arch            : arm64  (byte order: little)
python          : 3.13.0
cores / RAM     : 8 / None GB
disk free       : 29.9 GB
NNPA flag       : None
zDNN libs       : none found
ROOT            : /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk
threads setting : 3   demo event: fixture_event
----------------------------------------------------------------
  numpy              2.1.2
  pandas             2.2.3
  scikit-learn       1.7.0
  xgboost            NOT INSTALLED
  torch              2.9.0
  transformers       4.46.2
  tokenizers         0.20.3
  huggingface_hub    0.33.1
  llama_cpp_python   NOT INSTALLED
  ipywidgets         8.1.7
  matplotlib         3.10.3
  joblib             1.5.1
  torch_nnpa         NOT INSTALLED
  nbconvert          7.17.2
----------------------------------------------------------------
  --  humaid_dir     /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk/data/raw/humaid
  --  gazetteer_dir  /Users/sai

In [2]:
from surgedesk import config, fileio
from surgedesk.evaluate import measured_caption, evaluation_freshness, audit_privacy_artifacts
from IPython.display import display, Image, Markdown
import pandas as pd
D = config.OUTPUTS / config.DEMO_EVENT
def read_result(name):
    path = D / name
    return fileio.read_json(path) if path.exists() else {}
def show_records(records):
    if records:
        display(pd.DataFrame(records))
    else:
        print("not available yet")
def show_plot(name):
    path = D / "plots" / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"{name}: not available yet")
summary = read_result(config.SUMMARY_FILE)
if summary:
    freshness = evaluation_freshness(config.DEMO_EVENT, D)
    print(f"Evaluation status: {freshness['status']}. {freshness['note']}")
    if freshness.get("changed_inputs"):
        print("Changed inputs:", ", ".join(freshness["changed_inputs"]))
    print(measured_caption(summary["measured_on"]))
    print(f"Event: {summary["event"]}")
    show_records([summary["measured_on"]])
    provenance = read_result("label_provenance.json")
    if provenance:
        print(provenance.get("note", ""))
        if provenance.get("provisional"):
            print("PROVISIONAL AI-BASED EVALUATION; human validation pending.")
        show_records([provenance])
else:
    print("summary.json: not available yet; run python -m surgedesk.pipeline --stage eval")
# Recheck existing artifacts without rewriting labels, logs, or cached outputs.
privacy = audit_privacy_artifacts(config.DEMO_EVENT, D)
print("Artifact contact audit:", privacy["status"])
print(privacy["note"])
show_records(privacy["findings"] + privacy["unreadable"])


Development run, NOT LinuxONE
Event: fixture_event


,arch,cores,ram_gb,nnpa_flag
0,arm64,8,None,None


## Classifier

In [3]:
classifier = read_result("classifier_eval.json")
if classifier:
    show_records([dict(label=label, f1=f1) for label, f1 in classifier.get("per_class_f1", {}).items()])
    show_records([{k: classifier.get(k) for k in ("n", "accuracy", "macro_f1")}])
else:
    print("Classifier: not available yet")
show_plot("confusion.png")

Classifier: not available yet
confusion.png: not available yet


## Extraction

In [4]:
from surgedesk.contracts import BRIEF_FIELDS
extraction = summary.get("extraction", {})
if not extraction or extraction.get("status") == "skipped":
    print("awaiting reference labels / extraction results")
    if extraction.get("reason"):
        print(extraction["reason"])
else:
    show_records([dict(field=f, **extraction[f]) for f in BRIEF_FIELDS])
    show_records([{k: v for k, v in extraction.items() if k not in BRIEF_FIELDS}])

awaiting hand labels / extraction results
Awaiting hand labels and extraction_preds.jsonl (owner: Agent 3)


## Grouping and purity

In [5]:
grouping = read_result("grouping_metrics.json")
show_records([grouping] if grouping else [])
if summary:
    show_records([{k: summary.get(k) for k in ("n_reports", "n_actionable", "n_incidents", "singleton_share", "purity")}])

,status,reason,purity
0,skipped,group_stats.json not available yet (owner: Age...,None


,n_reports,n_actionable,n_incidents,singleton_share,purity
0,20,17,10,None,None


## Ranking and time to surface

In [6]:
show_records([dict(method=method, **metrics) for method, metrics in summary.get("ranking", {}).items()])
show_records([dict(method=method, **metrics) for method, metrics in summary.get("time_to_surface", {}).items()])
print("Time-to-surface medians exclude censored incidents; share surfaced includes them.")
show_plot("time_to_surface.png")
ranking = read_result("ranking_eval.json")
print("Ranking objective:", ranking.get("objective", "not available yet"))
print(ranking.get("target_note", ""))
show_records(ranking.get("target_comparison", []))
print("Compare methods within the same target. LegacySurgeDesk uses the preserved model on current groups when available.")


not available yet
not available yet
Time-to-surface medians exclude censored incidents; share surfaced includes them.
time_to_surface.png: not available yet


## Fairness

In [7]:
show_records(summary.get("fairness_demo_snapshot", []))
show_records(read_result("fairness_metrics.json").get("mean", []))
show_plot("fairness.png")

not available yet
not available yet
fairness.png: not available yet


## Briefings

In [8]:
show_records([summary["briefings"]] if "briefings" in summary else [])
review = read_result("incident_review.json")
if review:
    show_records([review])
    print("Masked source review:", D / "incident_review.csv")


,n,fields_grounded_share,sentences_accepted_share,fallback_share
0,3,0.866667,0.666667,0.666667


## Benchmark (run separately on an idle machine)

In [9]:
benchmark = read_result("benchmark_summary.json")
if benchmark:
    saved_objective = read_result("ranking_eval.json").get("objective")
    current_meta = fileio.read_json(config.RANKER_META_PATH) if config.RANKER_META_PATH.exists() else {}
    if benchmark.get("ranking_objective") != saved_objective or benchmark.get("ranker_trained_at") != current_meta.get("trained_at"):
        print("Benchmark does not match the current ranking objective/model. Rerun run_benchmark.sh on an idle machine before making current performance claims.")
    print(benchmark.get("caption", "not available yet"))
    show_records([{k: v for k, v in benchmark.items() if k not in ("measured_on", "caption")}])
else:
    print("run run_benchmark.sh")
for filename in ("benchmark_stream_summary.csv", "benchmark_granite.csv"):
    path = D / filename
    if path.exists():
        display(pd.read_csv(path))
    else:
        print(f"{filename}: not available yet; run run_benchmark.sh")
for filename in ("benchmark_granite_summary.json", "benchmark_embed.json"):
    result = read_result(filename)
    show_records([result] if result else [])
for name in ("benchmark_stream.png", "benchmark_granite.png"):
    show_plot(name)

Development run, NOT LinuxONE


,stream,granite,embed
0,"{'status': 'skipped', 'reason': 'stage stream ...","{'status': 'skipped', 'reason': 'stage Granite...","{'status': 'skipped', 'reason': 'stage embeddi..."


benchmark_stream_summary.csv: not available yet; run run_benchmark.sh
benchmark_granite.csv: not available yet; run run_benchmark.sh
not available yet


,status,reason
0,skipped,stage embedding device benchmark not implement...


benchmark_stream.png: not available yet
benchmark_granite.png: not available yet
